In [2]:
import os
from dotenv import load_dotenv
import mssql_python
import pyarrow as pa
import duckdb

load_dotenv()
conn = mssql_python.connect(os.getenv("MSSQL_CONNECTION_STRING"))
cursor = conn.cursor()

In [2]:
def query_arrow(sql):
    cursor.execute(sql)
    return cursor.arrow()

In [3]:
# Fetch a small, known set of zeros
tbl = query_arrow("""
    SELECT TOP 100 REACTIVA_H1
    FROM dbo.a
    WHERE REACTIVA_H1 = 0
""")

col = tbl.column("REACTIVA_H1")
print("rows:", tbl.num_rows)
print("arrow null_count:", col.null_count)
print("first 5 values:", col.to_pylist()[:5])

rows: 100
arrow null_count: 0
first 5 values: [0, 0, 0, 0, 0]


In [4]:
import os
import pyarrow as pa
import pyarrow.parquet as pq

os.makedirs("../data", exist_ok=True)

cursor.execute("SELECT * FROM dbo.a")

writer = None
total_rows = 0

for batch in cursor.arrow_reader(batch_size=100_000):
    # batch is a pyarrow.RecordBatch or pyarrow.Table
    if writer is None:
        writer = pq.ParquetWriter(
            "../data/a.parquet",
            schema=batch.schema,
            compression="snappy",
        )
    writer.write_batch(batch) if isinstance(batch, pa.RecordBatch) else writer.write_table(batch)
    total_rows += batch.num_rows
    print(f"  wrote {batch.num_rows} rows, total {total_rows}")

if writer is not None:
    writer.close()

print(f"Done. {total_rows} rows written to ../data/a.parquet")

  wrote 100000 rows, total 100000
  wrote 100000 rows, total 200000
  wrote 100000 rows, total 300000
  wrote 100000 rows, total 400000
  wrote 100000 rows, total 500000
  wrote 100000 rows, total 600000
  wrote 100000 rows, total 700000
  wrote 100000 rows, total 800000
  wrote 100000 rows, total 900000
  wrote 100000 rows, total 1000000
  wrote 100000 rows, total 1100000
  wrote 100000 rows, total 1200000
  wrote 100000 rows, total 1300000
  wrote 100000 rows, total 1400000
  wrote 100000 rows, total 1500000
  wrote 100000 rows, total 1600000
  wrote 100000 rows, total 1700000
  wrote 100000 rows, total 1800000
  wrote 100000 rows, total 1900000
  wrote 100000 rows, total 2000000
  wrote 100000 rows, total 2100000
  wrote 100000 rows, total 2200000
  wrote 100000 rows, total 2300000
  wrote 100000 rows, total 2400000
  wrote 100000 rows, total 2500000
  wrote 100000 rows, total 2600000
  wrote 100000 rows, total 2700000
  wrote 100000 rows, total 2800000
  wrote 100000 rows, total 29

In [10]:
tables_tbl = query_arrow("""
    SELECT TABLE_SCHEMA, TABLE_NAME
    FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_TYPE = 'BASE TABLE'
    ORDER BY TABLE_NAME
""")

# to_pylist() gives a list of dicts directly from the Arrow table
for row in tables_tbl.to_pylist():
    schema, table = row['TABLE_SCHEMA'], row['TABLE_NAME']
    print(f"\n{'=' * 60}")
    print(f"Table: {schema}.{table}")
    print('=' * 60)
    try:
        preview = query_arrow(f"SELECT TOP 5 * FROM [{schema}].[{table}]")
        print(preview)          # Arrow prints nicely on its own
    except Exception as e:
        print(f"  Skipped: {e}")


Table: dbo.a
pyarrow.Table
DIA: large_string
H1: int32
ACTIVA_H1: int32
REACTIVA_H1: int32
H2: int32
ACTIVA_H2: int32
REACTIVA_H2: int32
H3: int32
ACTIVA_H3: int32
REACTIVA_H3: int32
H4: int32
ACTIVA_H4: int32
REACTIVA_H4: int32
H5: int32
ACTIVA_H5: int32
REACTIVA_H5: int32
H6: int32
ACTIVA_H6: int32
REACTIVA_H6: int32
H7: int32
ACTIVA_H7: int32
REACTIVA_H7: int32
H8: int32
ACTIVA_H8: int32
REACTIVA_H8: int32
H9: int32
ACTIVA_H9: int32
REACTIVA_H9: int32
H10: int32
ACTIVA_H10: int32
REACTIVA_H10: int32
H11: int32
ACTIVA_H11: int32
REACTIVA_H11: int32
H12: int32
ACTIVA_H12: int32
REACTIVA_H12: int32
H13: int32
ACTIVA_H13: int32
REACTIVA_H13: int32
H14: int32
ACTIVA_H14: int32
REACTIVA_H14: int32
H15: int32
ACTIVA_H15: int32
REACTIVA_H15: int32
H16: int32
ACTIVA_H16: int32
REACTIVA_H16: int32
H17: int32
ACTIVA_H17: int32
REACTIVA_H17: int32
H18: int32
ACTIVA_H18: int32
REACTIVA_H18: int32
H19: int32
ACTIVA_H19: int32
REACTIVA_H19: int32
H20: int32
ACTIVA_H20: int32
REACTIVA_H20: int32
H

In [15]:
conn.close()

In [16]:
import pyarrow.parquet as pq
print(pq.ParquetFile("main_data.parquet").metadata.num_rows)  # should be 45,311,450

45311450
